# Rethinking Spatial Composite Indicators with the Lens of Machine Learning

**Spatial composite indicators** serve as concise performance metrics frequently employed in policy and decision-making contexts. They offer a streamlined depiction of **complex, multi-dimensional
systems** that would otherwise be challenging to grasp visually. 

They are used in various fields and some very famous examples are literacy rate, GDP, social vulnerability index, and air quality index.

However,it’s important to note that spatial composite indicators are 
inherently subjective, as the factors
considered in each indicator are determined by the decision-maker. Theoretically, one can incorporate numerous indicators, and here we would like to explore how it will help us to explore a wider set of alternatives for a given decison problem.

In this notebook, we will see a comprehensive approach for enhancing spatial composite indicators through the dimensional reduction capabilities of machine learning algorithms, particularly self-organizing maps. 

By using a case study on Arctic development, let's see how we can integrate diverse data sources, such as socioeconomic, environmental, and infrastructural datasets, can provide a more holistic perspective on spatial vulnerability and opportunity. 

## 1.   Getting the Decision Factors Ready and Defining the Zone of Interests

In this spatial decision making problem, we will consider various socio-economic , environmental, and infrastructure data which impacts Arctic development in various dimensions.  Below, you can see a table of example data gathered for this case study:

| Factor | Group | Data Source|
|---|---|---|
| Attribute 1 | Socio-economic | www.attribute1.org |
| Attribute 1 | Environmental | www.attribute1.org |
| Attribute 1 | Infrastructure | www.attribute1.org |

Since the location in Artic is very sparse, it would be good to have a fixed zone of interest to measure important statistics, in a way we can summarize the various factors. In this study we will use [H3 hexagons](https://www.uber.com/blog/h3/) (reslution 5) and summarize the decision factors within these hexagons. 

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns;sns.set()
from minisom import MiniSom
from sklearn.preprocessing import minmax_scale, scale
import time

In [ ]:
#here we can load the arctic boundary, create the hexagons, and do some zonal statistics with the updated data 
#for the time being I will use the clipped h3 hexagons and then generate random dataset

hexagons = gpd.read_file("clipped_h3_V2.shp")

# Check CRS and reproject if needed
print("Original CRS:", hexagons.crs)
if hexagons.crs != "EPSG:3413":
    hexagons = hexagons.to_crs("EPSG:3413")

# Plot the hexagons
fig, ax = plt.subplots(figsize=(10, 10))

hexagons.plot(ax=ax, facecolor="white", edgecolor="black", linewidth=1)

In [ ]:
# Set a random seed for reproducibility
#generating random pseudo dataset

np.random.seed(42)

#---------UPDATE THIS SECTION DEPENDING ON YOUR DATASET 
# Define the categories and the number of attributes per category
categories = [
    "demographics", 
    "animal_distribution", 
    "sea_level_rise", 
    "human_mobility", 
    "flood_risk", 
    "energy_infrastructure"
]
attributes_per_category = 10

# Generate random attributes for each category
for category in categories:
    for i in range(1, attributes_per_category + 1):
        col_name = f"{category}_attr{i}"
        # Generate random values (here, uniformly distributed between 0 and 100)
        hexagons[col_name] = np.random.rand(len(hexagons)) * 100

import time
timestr = time.strftime("%Y%m%d-%H%M%S") # this will be used during the output file generation

#---------UPDATE THIS SECTION DEPENDING ON YOUR DATASET 

#Drop any unnecessary attributes' names that won't be useful in SOM
feature_columns = list(hexagons.columns)[4:]
feature_names = feature_columns

#Drop any unnecesary attributes that won't be useful in SOM

hex_pdf = hexagons.drop(hexagons.columns[:4],axis=1)

print(str(len(hex_pdf))+ " features and " +str(len(hex_pdf.columns))+ " columns are added to the dataframe")

In [ ]:
X = hex_pdf[feature_names].values
X = scale(X)

feature_df = pd.DataFrame(X, columns=feature_names)
target = feature_df.iloc[:,0]
Features = feature_df.iloc[:,1:]

corrMatrix = feature_df.corr().round(2)
mask = np.zeros_like(corrMatrix)
mask[np.triu_indices_from(mask)] = True

with sns.axes_style("darkgrid"):
    f, ax = plt.subplots(figsize=(7, 7))
    ax = sns.heatmap(corrMatrix, mask=mask,center=0, linewidths=1, xticklabels=False)

plt.show()

In [ ]:
#if you want to provide a differen X now, that is the time! X=hex_pdf[feature_names].values

#SOM Parameters

size = 14 #size of the Unified Matrix
hc_function = 'gaussian'
kernel_width = 1
rnd_seed = 1
training_epoch = 100000

In [ ]:
som = MiniSom(size, size, len(X[0]),
              neighborhood_function=hc_function, sigma=kernel_width,
              random_seed=rnd_seed)

som.pca_weights_init(X)
som.train_random(X, training_epoch, verbose=True)

In [ ]:
SOM_indicies_map = som.labels_map(X, hex_pdf.index +1) #  

plt.figure(figsize=(14, 14))
for p, features in SOM_indicies_map.items():
    features = list(features)
    x = p[0] + .1
    y = p[1] - .3
    for i, c in enumerate(features):
        off_set = (i+1)/len(features) - 0.05
plt.pcolor(som.distance_map().T, cmap='gray_r', alpha=.5)
plt.xticks(np.arange(size+1))
plt.yticks(np.arange(size+1))
plt.grid()

plt.savefig("UMaxtrix_vector_"+str(timestr)+".svg", bbox_inches='tight', dpi=300)

plt.show()

In [ ]:
W= som.get_weights()
Z = np.zeros((size, size))
plt.figure(figsize=(8, 8))

clusterarray={}
color_hex =["#a6cee3","#1f78b4","#b2df8a","#33a02c","#fb9a99","#e31a1c","#fdbf6f","#ff7f00","#cab2d6","#6a3d9a",'#8dd3c7']
#color_hex= ['#8dd3c7','#ffffb3','#bebada','#fb8072','#80b1d3','#fdb462','#b3de69','#fccde5','#d9d9d9','#bc80bd']
#"tab20c" #"tab20b"
for i in np.arange(som._weights.shape[0]):
    for j in np.arange(som._weights.shape[1]):
        feature = np.argmax(W[i, j , :])
        clusterarray[i,j]=feature
        plt.plot([j+.5], [i+.5], 'o',
                 marker='s', markersize=24)

#legend_elements = [Patch(facecolor=color_hex[i],
#                         edgecolor='w',
#                         label=f) for i, f in enumerate(feature_names)]
legendlabels=[]

        
plt.xlim([0, size])
plt.ylim([0, size])
plt.savefig("clusters_vector_"+str(timestr)+".svg", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
cluster_xy_dic = pd.DataFrame(list(clusterarray.items()),columns=['UM_Location','ClusterID'])
cluster_xy_dic


In [ ]:
feature_xy_dic = pd.DataFrame(list(SOM_indicies_map.items()),columns=['UM_Location','Features'])
feature_xy_dic

In [ ]:
joined = pd.merge(cluster_xy_dic,feature_xy_dic)
joined

In [ ]:
split_df = pd.DataFrame(joined['Features'].tolist())
df = pd.concat([joined, split_df], axis=1)
df

In [ ]:
rows=[]
featurecodes=hex_pdf.index +1  #if that is something different than the OID replace this!

for feature in featurecodes:
    clusterID = df.loc[df[feature] == 1.0, 'ClusterID'].iloc[0]
    row = [feature,clusterID]
    rows.append(row)

thematic_df = pd.DataFrame(rows,columns=["Feature","ClusterID"])
thematic_df["ClusterID"]=thematic_df["ClusterID"]+1
thematic_df

# Spatialization

In [ ]:
import geopandas as gpd
hexagons.head()
hexagons ["OID"]=hexagons.index+1
hexagons.head()

In [ ]:
df = hexagons.merge(thematic_df, left_on='OID', right_on='Feature' , how='left')
gdf =gpd.GeoDataFrame(df)
gdf.head()

In [ ]:
gdf.plot(column='ClusterID',legend=True,cmap='Spectral')


In [ ]:
fig, ax = plt.subplots(1, figsize=(14,8))
gdf.plot(column='ClusterID', categorical=True, cmap='Spectral', linewidth=.6, edgecolor='0.2',
         legend=True, legend_kwds={'bbox_to_anchor':(1.05, 0.55),'fontsize':16,'frameon':False}, ax=ax)
ax.axis('off')
ax.set_title('Most Influential Indicies based on SOM Clusters',fontsize=20)
plt.tight_layout()